# Data Exploration for LLM-as-a-Fuzzy-Judge

This notebook explores the Excel files containing labeled clinical interactions for fine-tuning an LLM as a fuzzy clinical evaluation judge based on:

- **Professionalism**: {1. Unprofessional, 2. Borderline, 3. Appropriate}
- **Medical Relevance**: {1. Irrelevant, 2. Partially relevant, 3. Relevant}
- **Ethical Behavior**: {1. Dangerous, 2. Unsafe, 3. Questionable, 4. Mostly safe, 5. Safe}
- **Contextual Distraction**: {1. Highly distracting, 2. Moderately distracting, 3. Questionable, 4. Not distracting}

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# For better plot display
%matplotlib inline
# Ensure you have a valid style, like 'seaborn-v0_8-whitegrid' if using newer matplotlib/seaborn
try:
    plt.style.use('seaborn-v0_8-whitegrid') 
except OSError:
    plt.style.use('seaborn-whitegrid') # Fallback
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 8]

# Adjust pandas display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

: 

## 1. Load Raw Data Files

In [ ]:
# Set the path to the raw data directory (relative to the notebook location)
raw_data_dir = Path('../data/raw')

# Get list of Excel files
excel_files = sorted(list(raw_data_dir.glob('fuzzy.coding.data*.xlsx')))
print(f"Found {len(excel_files)} Excel files in {raw_data_dir}:")
for file in excel_files:
    print(f"- {file.name}")

In [ ]:
# Load each Excel file into a dictionary of DataFrames
data_dict = {}
for file_path in excel_files:
    try:
        # Load only the first sheet by default, adjust if needed
        df = pd.read_excel(file_path)
        data_dict[file_path.name] = df
        print(f"Loaded {file_path.name}: {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Error loading {file_path.name}: {e}")

## 2. Examine Data Structure and Schema

Let's look at the structure of the first DataFrame and check for consistency across files.

In [ ]:
# Get the first DataFrame
if data_dict:
    first_file = next(iter(data_dict))
    first_df = data_dict[first_file]
    
    print(f"--- Examining file: {first_file} ---")
    print("
Columns:")
    print(list(first_df.columns))
    
    print("
Data types:")
    print(first_df.dtypes)
    
    print("
First 5 rows:")
    display(first_df.head())
    
    print("
Last 5 rows:")
    display(first_df.tail())
else:
    print("No data available.")

In [ ]:
# Check schema consistency
if data_dict and len(data_dict) > 1:
    print("--- Checking schema consistency across files ---")
    first_cols = set(first_df.columns)
    consistent = True
    for file_name, df in data_dict.items():
        current_cols = set(df.columns)
        if current_cols != first_cols:
            print(f"Schema mismatch in {file_name}:")
            missing = first_cols - current_cols
            extra = current_cols - first_cols
            if missing: print(f"  Missing columns: {missing}")
            if extra: print(f"  Extra columns: {extra}")
            consistent = False
    if consistent:
        print("All files have the same columns.")
else:
    print("Only one file loaded, no schema comparison needed.")

## 3. Merge DataFrames

Merge all data into a single DataFrame for unified analysis.

In [ ]:
# Merge all DataFrames, adding a 'source_file' column
if data_dict:
    all_dfs = []
    for file_name, df in data_dict.items():
        df = df.copy() # Avoid SettingWithCopyWarning
        df['source_file'] = file_name
        all_dfs.append(df)
        
    merged_df = pd.concat(all_dfs, ignore_index=True)
    print(f"Merged DataFrame shape: {merged_df.shape}")
    print("Columns in merged DataFrame:", list(merged_df.columns))
    display(merged_df.head())
else:
    print("No data to merge.")

## 4. Identify Key Columns

Based on the column names and data types, let's try to identify the main text column (e.g., student interaction) and the four label columns for our fuzzy criteria.

In [ ]:
# **ACTION REQUIRED:** 
# Based on the output of the previous cells (column names, data types, head/tail view),
# please specify the exact column names below.

# Example: 
# text_column = 'student_utterance'
# professionalism_column = 'Prof_Level' 
# relevance_column = 'Med_Rel'
# ethics_column = 'Ethical_Behav'
# distraction_column = 'Context_Dist'

text_column = 'FILL_IN_TEXT_COLUMN' # Replace with the actual column name for the interaction text
professionalism_column = 'FILL_IN_PROFESSIONALISM_COLUMN' # Replace with the actual column name
relevance_column = 'FILL_IN_RELEVANCE_COLUMN' # Replace with the actual column name
ethics_column = 'FILL_IN_ETHICS_COLUMN' # Replace with the actual column name
distraction_column = 'FILL_IN_DISTRACTION_COLUMN' # Replace with the actual column name

# Store these in a dictionary for later use
label_columns = {
    'professionalism': professionalism_column,
    'relevance': relevance_column,
    'ethics': ethics_column,
    'distraction': distraction_column
}

# Verify if these columns exist in the merged DataFrame
if 'merged_df' in locals():
    all_found_cols = [text_column] + list(label_columns.values())
    missing_cols = [col for col in all_found_cols if col.startswith('FILL_IN_') or col not in merged_df.columns]
            
    if not missing_cols:
        print("Successfully identified specified columns:")
        print(f"  Text Column: {text_column}")
        for criterion, col_name in label_columns.items():
            print(f"  {criterion.capitalize()} Column: {col_name}")
    else:
        print(f"Error: One or more specified columns were not found or not replaced in the DataFrame: {missing_cols}")
        print("Please update the 'FILL_IN...' placeholders in this cell and re-run.")
else:
    print("Merged DataFrame not available.")

## 5. Analyze Label Distributions

Let's examine the distribution of values within each identified label column.

In [ ]:
# Proceed only if columns are specified and exist
proceed_analysis = False
if 'merged_df' in locals() and text_column != 'FILL_IN_TEXT_COLUMN':
    all_found_cols = [text_column] + list(label_columns.values())
    missing_cols = [col for col in all_found_cols if col.startswith('FILL_IN_') or col not in merged_df.columns]
    if not missing_cols:
        proceed_analysis = True

if proceed_analysis:
    print("--- Analyzing Label Distributions ---")
    plt.figure(figsize=(15, 12))
    plot_idx = 1
    
    for criterion, col_name in label_columns.items():
        print(f"
--- {criterion.capitalize()} ({col_name}) ---")
        # Drop NaN values for value_counts
        value_counts = merged_df[col_name].dropna().value_counts().sort_index()
        print("Value Counts:")
        print(value_counts)
        
        # Plot distribution
        plt.subplot(2, 2, plot_idx)
        sns.countplot(data=merged_df.dropna(subset=[col_name]), x=col_name, order=value_counts.index)
        plt.title(f'{criterion.capitalize()} Distribution')
        plt.xlabel('Label')
        plt.ylabel('Count')
        plt.xticks(rotation=30)
        plot_idx += 1
            
    plt.tight_layout()
    plt.show()
else:
    print("Skipping label analysis because column names are not specified or data is missing.")

## 6. Define Label Mappings

Define the mapping from the observed label values (strings or numbers) to the numerical indices (0, 1, 2...) required for model training.

In [ ]:
# **ACTION REQUIRED:** 
# Based on the 'Value Counts' output above, define the mappings.
# Ensure the keys in the mapping match the exact values found in your columns.

# Example (if labels are strings):
# label_maps = {
#     professionalism_column: {'Unprofessional': 0, 'Borderline': 1, 'Appropriate': 2},
#     relevance_column: {'Irrelevant': 0, 'Partially relevant': 1, 'Relevant': 2},
#     ethics_column: {'Dangerous': 0, 'Unsafe': 1, 'Questionable': 2, 'Mostly safe': 3, 'Safe': 4},
#     distraction_column: {'Highly distracting': 0, 'Moderately distracting': 1, 'Questionable': 2, 'Not distracting': 3}
# }

# Example (if labels are numbers like 1, 2, 3 - mapping to 0-based index):
# label_maps = {
#     professionalism_column: {1: 0, 2: 1, 3: 2}, # Map 1->0, 2->1, 3->2
#     relevance_column: {1: 0, 2: 1, 3: 2},
#     ethics_column: {1: 0, 2: 1, 3: 2, 4: 3, 5: 4},
#     distraction_column: {1: 0, 2: 1, 3: 2, 4: 3} 
# }

label_maps = {
    # Use the actual column names identified in step 4
    professionalism_column: { 
        # Fill in observed_value: desired_index pairs, e.g.: 
        # 'Unprofessional': 0, 'Borderline': 1, 'Appropriate': 2 
    },
    relevance_column: { 
        # 'Irrelevant': 0, 'Partially relevant': 1, 'Relevant': 2 
    },
    ethics_column: { 
        # 'Dangerous': 0, 'Unsafe': 1, 'Questionable': 2, 'Mostly safe': 3, 'Safe': 4 
    },
    distraction_column: { 
        # 'Highly distracting': 0, 'Moderately distracting': 1, 'Questionable': 2, 'Not distracting': 3 
    }
}

# Clean up the map dictionary keys (use actual column names)
# This step assumes the variables like professionalism_column hold the correct names
# We filter out any entries where the key is still the placeholder 'FILL_IN...'
defined_label_maps = {k: v for k, v in label_maps.items() if k is not None and not k.startswith('FILL_IN_')}

print("Defined Label Mappings (excluding placeholders):")
if defined_label_maps:
    for col, mapping in defined_label_maps.items():
        print(f"  Column '{col}': {mapping}")
        # Basic validation
        num_classes = len(mapping)
        expected_indices = set(range(num_classes))
        actual_indices = set(mapping.values())
        if expected_indices != actual_indices:
            print(f"    Warning: Indices for {col} are not sequential starting from 0! Found: {actual_indices}")
else:
    print("No valid label maps defined yet. Please fill in column names and mappings.")

## 7. Check Missing Values

Examine missing values, especially in the text and label columns.

In [ ]:
if proceed_analysis: # Use the flag set after column identification
    print("--- Checking Missing Values ---")
    missing_values = merged_df.isnull().sum()
    missing_percent = (missing_values / len(merged_df)) * 100
    
    missing_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage (%)': missing_percent
    })
    
    # Filter to show only columns with missing values
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
    print("Missing values per column (descending):")
    display(missing_df)
    
    # Specifically check the key columns
    key_cols = [text_column] + list(label_columns.values())
    print("
Missing values in key columns:")
    display(missing_df.loc[missing_df.index.intersection(key_cols)])
    
    # Calculate rows to be potentially dropped
    rows_before = len(merged_df)
    rows_after = len(merged_df.dropna(subset=key_cols))
    rows_dropped = rows_before - rows_after
    percent_dropped = (rows_dropped / rows_before) * 100 if rows_before > 0 else 0
    print(f"
If we drop rows with missing values in any key column ({key_cols}):")
    print(f"  Original rows: {rows_before}")
    print(f"  Rows after dropping NaN: {rows_after}")
    print(f"  Rows dropped: {rows_dropped} ({percent_dropped:.2f}%)")
else:
    print("Skipping missing value analysis because column names are not specified or data is missing.")

## 8. Analyze Text Length

Analyze the distribution of text lengths in the identified text column. This helps in choosing an appropriate `max_length` for the tokenizer.

In [ ]:
if proceed_analysis: # Use the flag set after column identification
    print(f"--- Analyzing Text Length ({text_column}) ---")
    # Ensure text is string and calculate word counts
    merged_df[text_column] = merged_df[text_column].astype(str)
    word_counts = merged_df[text_column].apply(lambda x: len(x.split()))
    
    plt.figure(figsize=(12, 6))
    sns.histplot(word_counts, bins=50, kde=False)
    plt.title(f'Distribution of Word Counts in {text_column}')
    plt.xlabel('Number of Words')
    plt.ylabel('Frequency')
    plt.show()
    
    print("Summary Statistics for Word Count:")
    print(word_counts.describe(percentiles=[.5, .75, .90, .95, .99]))
    
    # Suggest a max_length based on percentile (e.g., 95th)
    suggested_max_length = int(word_counts.quantile(0.95))
    # Often align with transformer model limits (e.g., 512)
    final_suggestion = min(suggested_max_length, 512) 
    print(f"
Suggested max_length for tokenizer based on 95th percentile (capped at 512): {final_suggestion}")
else:
    print("Skipping text length analysis because column names are not specified or data is missing.")

## 9. Exploration Summary & Next Steps

**Summary:**
*   **ACTION REQUIRED:** Fill in the summary based on the notebook outputs above.
1.  **Key Columns Identified:**
    - Text Column: `[...]`
    - Professionalism: `[...]`
    - Relevance: `[...]`
    - Ethics: `[...]`
    - Distraction: `[...]`
2.  **Label Values & Mappings:** [Describe observed labels and confirm mappings defined in step 6]
3.  **Missing Values:** [Summarize findings, especially for key columns, and impact of dropping rows]
4.  **Text Length:** [Summarize findings, note suggested `max_length` for tokenizer]

**Next Steps:**
1.  Update `src/data/data_loader.py` with the identified columns and label mappings.
2.  Update `main.py` to accept the four label columns as arguments.
3.  Refactor `main.py` to train separate models for each criterion.
4.  Update documentation.